# **Lab:** Security in Computer Networks

**Student name:** write your name here

### **Topics**
- Confidentiality
- Public Key Cryptography (RSA)
- Digital Signatures
- TLS Handshake
- Attack Simulation (CTF)

In [1]:
# Install cryptography library (only first time)
# This library provides tools for encryption, decryption, signatures, etc.
%pip install cryptography

  Using cached cffi-2.0.0-cp313-cp313-macosx_10_13_x86_64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 8.9 MB/s  0:00:01m0:00:0100:01
Using cached cffi-2.0.0-cp313-cp313-macosx_10_13_x86_64.whl (185 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [cryptography] [cryptography]
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Import RSA module for asymmetric cryptography
from cryptography.hazmat.primitives.asymmetric import rsa

# Import serialization tools (used for saving/loading keys, not used deeply here)
from cryptography.hazmat.primitives import serialization

# Import padding schemes (needed for secure encryption)
from cryptography.hazmat.primitives.asymmetric import padding

# Import hashing algorithms (SHA256, etc.)
from cryptography.hazmat.primitives import hashes

## **🔐 Part 1:** RSA (Public Key Cryptography)

- Public key → shared
- Private key → secret
- Encrypt with public key, decrypt with private key


In [7]:
# Generate PRIVATE key
# public_exponent=65537 is a standard safe value used in RSA
# key_size=2048 means 2048-bit key (secure for most purposes)
private_key = rsa.generate_private_key(
    public_exponent=65537,   # mathematical parameter used in RSA
    key_size=2048            # size of the key in bits (larger = more secure but slower)
)

# Generate PUBLIC key from private key
# In RSA, both keys are mathematically linked
public_key = private_key.public_key()

print("Keys generated successfully")

Keys generated successfully


In [ ]:
import base64

# Define message to encrypt
# .encode() converts string → bytes (required for crypto functions)
message = "Hello secure world".encode()

# Encrypt using PUBLIC key
encrypted = public_key.encrypt(
    message,  # data to encrypt (must be bytes)
    padding.OAEP(  # OAEP = Optimal Asymmetric Encryption Padding (secure padding)
        mgf=padding.MGF1(algorithm=hashes.SHA256()),  # MGF1 = Mask Generation Function
        algorithm=hashes.SHA256(),  # hash algorithm used internally
        label=None  # optional label (not needed here)
    )
)

print("Encrypted (Python bytes object):", encrypted) # This will show as a bytes object (not human-readable). ASCIIs are not suitable for binary data, so we will convert it to hex and base64 for display.
print("Encrypted (hex):", encrypted.hex())
print("Encrypted (base64):", base64.b64encode(encrypted).decode()) # Standard way to represent binary data as text

Encrypted (Python bytes object): b'v\x90\xa5{\xbdg\xa7\x85\xb3\x11\xaf\xb8\'\xf5\xb8\xccY\x81Q\xf1s\x0b\x18l\xd4/f,\x02\xb0\xbb\x9c\x9f\xf7\xa1\xbd\xccqwL\xa6\x0b\xb9\xe2\xe7<\x03z\xaf\xd3\\\x9d\x87\xb6\xe5Z"0[}2/\xb9\x8e\xd5\xee\xf1\x10Z\xf5}\xf7\n\xc0i1GF\xf7+#\x8f\x19\xdc\xb7\xb7=5c\xa3\xef\x1b>\xc1\xb4\x9f\xd04,A\xca\xce-\xb0K\xbb\xe7\x8bB\xc4]\x1a\xff\x10\x80W\xa8\x0b\xad\x17\r\xd9\x93\x87\xc5Y\x04\xec\xb5q\xd8KV\xdd\x96\x0f\x03LI\x98\xbdL\xc9(}\x88\xe0a>"[\xe4\xd3W\x8b`\x80![\x83+\x11\x11\xae\x9bH\x11\xdb\xa4\xb6\xfe\x0f\x9e\xc9\xb8\x9a\x15\xee\xf1\x9c\x06L\xa5\xccy\xe1\x98\x86\x87\xb2n\r\x81\xd3\x00B\x91\x8bJ\xa9\x8d\xa5\x12\xcd\xba\x96\xf59\rJf\xdd\xdc\xecyb\x7f0;\xae\xbbK\xfd\xe7I\x95\xa3AI\x05\x19\xb3\x14\x07\x0b\xca\xb2\xd3\xa6\xd6o\x85\x8e\xcb\x9f\xa4x\xdd\x91\xaf\no\xa50\xf3f'
Encrypted (hex): 7690a57bbd67a785b311afb827f5b8cc598151f1730b186cd42f662c02b0bb9c9ff7a1bdcc71774ca60bb9e2e73c037aafd35c9d87b6e55a22305b7d322fb98ed5eef1105af57df70ac069314746f72b238f19dcb7b73d3563a3ef

In [15]:
# Decrypt using PRIVATE key
decrypted = private_key.decrypt(
    encrypted,  # encrypted data
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Convert bytes → string using .decode()
print("Decrypted:", decrypted.decode())

Decrypted: Hello secure world


When you:

1. Take a message  
2. Encrypt it with the **receiver’s public key**  
3. Encode it in **Base64**

You already achieved **confidentiality**:  
Only the holder of the **private key** can decrypt it.

> ⚠️ **Base64 is NOT security**, it is just an encoding format to represent binary data as text.

The real problem (without certificates)

The key question is:

> ❗ How do you know that the **public key** actually belongs to the right person?

Without certificates, this can happen:

- An attacker gives you **their public key**
- You encrypt the message using that key
- The attacker decrypts it with their private key
- Then re-encrypts it using the real recipient’s key

*Result:* your communication was intercepted without you noticing.

This is known as a **Man-in-the-Middle** attack.

So… what are certificates for?

A certificate (e.g., `X.509`) is used to:

**Bind:**
- A **public key**
- To a **real identity** (person, server, organization)

**It also includes:**
- A trusted signature (from a Certificate Authority)
- Owner information
- Expiration/validity dates

Who guarantees this?

A **Certificate Authority (CA)** such as:

- Let's Encrypt  
- DigiCert  

These entities sign certificates and essentially say:

> “Yes, this public key belongs to this entity.”

## **✍️ Part 2:** Digital Signatures

- Sign with PRIVATE key
- Verify with PUBLIC key


In [16]:
# Create digital signature
signature = private_key.sign(
    message,  # original message
    padding.PSS(  # Probabilistic Signature Scheme (secure signature padding)
        mgf=padding.MGF1(hashes.SHA256()),  # mask generation function
        salt_length=padding.PSS.MAX_LENGTH  # random salt for security
    ),
    hashes.SHA256()  # hash algorithm used before signing
)

print("Signature created")

Signature created


In [17]:
# Verify signature
try:
    public_key.verify(
        signature,  # signature to verify
        message,    # original message
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Signature VALID")
except:
    print("Signature INVALID")

Signature VALID


How it fits together?

In protocols like **Transport Layer Security (TLS)**:

1. The server sends its certificate  
2. The client verifies:
   - The CA signature  
   - The domain name  
3. **Only then** the public key is trusted  
4. The key is used for encryption

## **🌐 Part 3:** TLS Handshake Simulation

Real TLS:
1. Client gets server public key
2. Client creates session key
3. Client encrypts session key with public key
4. Server decrypts
5. Both use symmetric encryption


In [18]:
# Import symmetric encryption tool
from cryptography.fernet import Fernet

In [19]:
# Client generates symmetric session key
# This key will be used for fast encryption (like AES)
session_key = Fernet.generate_key()

# Create cipher object for encryption/decryption
cipher_client = Fernet(session_key)

print("Session key created")

Session key created


In [20]:
# Client encrypts session key using server PUBLIC key
encrypted_session_key = public_key.encrypt(
    session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

In [21]:
# Server decrypts session key using PRIVATE key
decrypted_session_key = private_key.decrypt(
    encrypted_session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Server creates cipher with shared key
cipher_server = Fernet(decrypted_session_key)


In [22]:
# Secure communication
secure_message = cipher_client.encrypt(b"Hello TLS secure world")

print("Encrypted message:", secure_message)

print("Decrypted message:", cipher_server.decrypt(secure_message).decode())

Encrypted message: b'gAAAAABp565zB0hTUrUNtIgq4jh8iK6TECgPBk3TUJiqY56jiJAz2H3EwKi9H8cW3UGhbQY5UFo7SsdjBskuETPyp8J_qZOB7MCSQeSembfxPPdNbzesG5c='
Decrypted message: Hello TLS secure world


## **Assignment:** Secure Communication Challenge

### **Objective**
Apply all concepts from the lab:
- Confidentiality (encryption)
- Authentication (digital signatures)
- Integrity (verification)

### **Group Dynamics (Balanced Work)**

Each group must complete ALL roles:

1. **Sender**
2. **Receiver**
3. **Attacker**

You will interact with at least **two other groups**.

### **Task 1:** Secure Message Sending

Each group must:

1. Generate an RSA key pair
2. Share ONLY the **public key**
3. Create a message
4. Encrypt the message using the receiver’s public key
5. Sign the message using your private key

👉 Send to another group:
- Encrypted message
- Signature
- Your public key

### **Task 2:** Secure Message Receiving

Each group must:

1. Receive:
   - Encrypted message
   - Signature
   - Sender's public key

2. Decrypt the message using your private key  
3. Verify the signature using the sender’s public key  

✔ If valid → accept message  
❌ If invalid → reject message  

### **Time Limit**
- 20–25 minutes total

### **Evaluation (Checklist)**

| Criteria | ✔ / ✘ |
|--------|------|
| RSA encryption implemented | ✔ / ✘ |
| Digital signature implemented | ✔ / ✘ |
| Correctly decrypted & verified message | ✔ / ✘ |
| Explanation reflection questions | ✔ / ✘ |

### **Reflection Questions**

1. Why is encryption alone NOT enough for security?  
2. What is the role of digital signatures?  
3. Why does TLS use a hybrid approach?  